# Qampaign customer selection workbook

This notebook runs the complete pipeline from the organizer workbook and reviews its saved outputs. Run from the `Qampaign` directory. The Stage 1 test set has no labels and is used only for prediction; preprocessing, model selection, and calibration are fitted within training folds.

## 1. Preprocessing and data checks

We load the organizer workbook, validate IDs and constraints, exclude `duration`, and report the training base rate. Numeric and categorical transformations are fitted within each CV training fold by the Stage 1 pipeline.

In [1]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
from src.customer_pipeline.workbook import load_participant_workbook
OUT = ROOT / 'results'
data = load_participant_workbook(ROOT / 'data/bank_campaign_participant_release.xlsx')
train, test = data['train'], data['test']
print(f"Train: {len(train)} rows; test: {len(test)} rows; positive rate: {data['train_base_rate']:.1%}")
print(f"ID mapping: {data['mapping_status']} — {data['mapping_note']}")
print('duration present:', 'duration' in train.columns or 'duration' in test.columns)
display(train.head())
display(pd.DataFrame([data['constraints']], index=['workbook constraints']))

Train: 300 rows; test: 100 rows; positive rate: 11.7%
ID mapping: WORKBOOK_IDS — ID mapping check: this workbook contains explicit Stage1_Test customer_id values; they are used directly, verified unique, disjoint from train, and present in Stage2_Data.
duration present: False


,customer_id,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,C0273,51,technician,married,primary,no,1550,no,no,unknown,20,jun,5,-1,0,unknown,0
1,C0353,51,self-employed,married,secondary,yes,34,no,yes,cellular,7,jul,1,-1,0,unknown,0
2,C0201,32,services,married,tertiary,no,4908,no,yes,cellular,13,may,2,-1,0,unknown,0
3,C0014,43,blue-collar,married,secondary,no,10910,yes,no,cellular,15,may,2,-1,0,unknown,0
4,C0308,37,management,married,tertiary,no,5355,no,no,telephone,17,feb,4,-1,0,unknown,1


,N,K,C,H
workbook constraints,10,6,25,22


## 2. Classical and quantum baselines

The nested repeated stratified CV comparison uses identical folds for regularized logistic regression and a ZZ quantum-kernel SVC. Report ROC-AUC, PR-AUC, Brier score, and log loss with the repeated-CV intervals. The calibration plot compares pooled out-of-fold probabilities with observed outcomes.

In [2]:
# This is the end-to-end pipeline run. Its outputs are saved under results/ and at the repository root.
from src.customer_pipeline.pipeline import run_pipeline
if (OUT / 'stage1_cv_metrics.json').exists() and (OUT / 'stage2_selection.json').exists() and (OUT / 'stage3_simulation.json').exists():
    print('Using existing complete pipeline outputs in results/. Delete them to rerun the full pipeline.')
else:
    run_pipeline('settings.yaml')

2026/11/11 12:11:59 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at D:\python_download\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Using existing complete pipeline outputs in results/. Delete them to rerun the full pipeline.


In [3]:
cv = json.loads((OUT / 'stage1_cv_metrics.json').read_text(encoding='utf-8'))
rows = []
for family, result in cv['candidate_results'].items():
    summary = result['summary']
    row = {'model': family}
    for metric in ('roc_auc', 'pr_auc', 'brier', 'log_loss'):
        item = summary[metric]
        row[metric] = item['mean']
        row[f'{metric} 95% CI'] = f"{item['ci95_low']:.3f}–{item['ci95_high']:.3f}"
    rows.append(row)
display(pd.DataFrame(rows).set_index('model').round(3))
cal = pd.DataFrame(cv['calibration_curve'])
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax.plot(cal['mean_predicted'], cal['observed_rate'], marker='o', label='Pooled OOF')
ax.set(xlabel='Mean predicted probability', ylabel='Observed positive fraction', title='Stage 1 calibration')
ax.grid(alpha=.25); ax.legend(); plt.tight_layout()

,roc_auc,roc_auc 95% CI,pr_auc,pr_auc 95% CI,brier,brier 95% CI,log_loss,log_loss 95% CI
model,,,,,,,,
nested_logistic,0.619,0.577–0.661,0.226,0.193–0.259,0.097,0.096–0.099,0.342,0.336–0.348
nested_quantum,0.608,0.582–0.633,0.149,0.131–0.167,0.102,0.101–0.103,0.354,0.349–0.359


## 3. QUBO objective and Dicke/XY formulation

For candidate bit vector `x`, maximize `sum_i (p_i * value_i - cost_i) x_i`, subject to exactly six selected and the budget and capacity limits. The alternative slack QUBO minimizes negative net value plus penalty-weighted squared residuals for the cardinality equality and each resource inequality. The Dicke/XY QAOA instead starts in the uniform weight-six Dicke state and uses a complete-graph XY mixer, preserving cardinality; budget and capacity are checked when bitstrings are decoded. Exact enumeration checks all 210 six-of-ten choices and supplies the submitted selection.

In [4]:
stage2 = json.loads((OUT / 'stage2_selection.json').read_text(encoding='utf-8'))
print('Objective:', stage2['objective'])
print('QUBO:', json.dumps(stage2['qubo'], indent=2))
print('Exact:', json.dumps(stage2['exact'], indent=2))

Objective: sum(predicted_probability * expected_deposit_value - contact_cost)
QUBO: {
  "decision_variables": 10,
  "alternative_status": "20-qubit slack QUBO retained as an alternative encoding; its penalty minimum is checked against the exact subset optimum.",
  "budget_slack_variables": 5,
  "capacity_slack_variables": 5,
  "penalty_weight": 10000.0,
  "minimum_energy_over_decision_vectors": -16.122051120975613,
  "matches_exact_bruteforce_optimum": true,
  "constraint_encoding": "Exactly-K uses penalty*(sum(x)-K)^2. Each <= resource limit uses penalty*(sum(resource_i*x_i)+binary_slack-limit)^2; binary slack bits represent 0 through the limit."
}
Exact: {
  "combinations_checked": 210,
  "feasible_combinations": 210,
  "selected_ids": [
    "C0081",
    "C0031",
    "C0079",
    "C0222",
    "C0087",
    "C0256"
  ],
  "bitstring_x0_to_xN": "1110100101",
  "decoding": "left-to-right bit i corresponds to top-10 rank i+1 and that row's customer_id",
  "objective_value": 16.12205112097

## 4. Stage 2 QAOA circuit and convergence

The circuit below uses the best measured-mass configuration recorded in the run. Its initial state has Hamming weight six; each layer applies objective phase rotations and pairwise XX/YY mixer gates.

In [5]:
from IPython.display import display
from src.customer_pipeline.stage2 import _qaoa_circuit
run = stage2['qaoa']['best_optimization_run']
candidates = pd.DataFrame(stage2['candidate_rows'])
net = candidates['expected_net_value'].to_numpy(float)
p = int(run['p']); theta = np.asarray(run['parameters'], dtype=float)
qaoa_circuit = _qaoa_circuit(net, p, theta[:p], theta[p:], measure=True)
display(qaoa_circuit.draw('mpl', fold=60))
conv = np.asarray(stage2['qaoa']['best_convergence'], dtype=float)
plt.figure(figsize=(7, 4)); plt.plot(np.arange(1, len(conv)+1), -conv)
plt.xlabel('Objective evaluation'); plt.ylabel('Best normalized expected objective')
plt.title(f"Dicke/XY QAOA convergence ({run['optimizer']}, p={p})"); plt.grid(alpha=.25); plt.tight_layout()
print(f"Best-of-80 exact-optimum sampled mass: {stage2['qaoa']['exact_optimum_probability_mass']:.1%}")

<Figure size 5136.94x4966.5 with 1 Axes>

Best-of-80 exact-optimum sampled mass: 37.9%


## 5. Stage 3 circuit, measured counts, and exact probability

Each selected customer is represented by an RY rotation whose measurement probability is that customer's Stage 1 prediction. Success means at least three of six measured bits are one. The chart compares the shot counts with the exact Poisson-binomial success probability computed from the same six probabilities.

In [6]:
stage3 = json.loads((OUT / 'stage3_simulation.json').read_text(encoding='utf-8'))
from qiskit import QuantumCircuit
angles = stage3['circuit_parameters']['angles_radians']
circuit3 = QuantumCircuit(6, 6)
for q, angle in enumerate(angles):
    circuit3.ry(angle, q); circuit3.measure(q, q)
display(circuit3.draw('mpl'))
counts = stage3['counts']
observed = sum(v for bits, v in counts.items() if sum(int(c) for c in bits.replace(' ', '')) >= 3) / stage3['shots']
exact = stage3['exact_probability_from_same_six_probabilities']
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Measured shots', 'Exact'], [observed, exact], color=['#4C78A8', '#F58518'])
ax.set_ylim(0, max(observed, exact)*1.3); ax.set_ylabel('P(success: at least 3 of 6)')
ax.set_title(f"Stage 3: counts vs exact ({stage3['shots']:,} shots)")
for i, value in enumerate([observed, exact]): ax.text(i, value, f'{value:.4f}', ha='center', va='bottom')
plt.tight_layout()
print('Success shots:', stage3['success_shots'], '/', stage3['shots'])
print('Exact Poisson-binomial probability:', f'{exact:.6f}')

<Figure size 705.552x618.722 with 1 Axes>

Success shots: 3620 / 20000
Exact Poisson-binomial probability: 0.181754
